# LINK VERIFY AI — Phishing URL Detection Model Training & Evaluation

This notebook documents the end-to-end dataset validation, static lexical feature extraction, leakage-free splitting, baseline and ensemble model comparison, and held-out test evaluation for **LINK VERIFY AI**.

## Objectives
1. Load and audit the **Hannousse & Yahiouche (2021)** public phishing URL benchmark dataset (`pirocheto/phishing-url`, 11,430 balanced URLs).
2. Deduplicate normalized URLs **before** splitting to eliminate train/validation/test leakage.
3. Extract purely static lexical & structural URL features (matching `backend/app/services/feature_extractor.py`) with zero live webpage fetching.
4. Compare **Majority Baseline**, **Logistic Regression**, **Random Forest**, and **XGBoost** on identical validation splits.
5. Evaluate the selected model once on the held-out test split (Accuracy, Precision, Recall, F1-Score, ROC-AUC, Confusion Matrix).

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().resolve().parent.parent
sys.path.insert(0, str(ROOT / 'ml-training' / 'training'))
sys.path.insert(0, str(ROOT / 'backend'))

from fetch_dataset import VALIDATED_CSV, download_benchmark_dataset, validate_and_normalize_dataset
from preprocess import build_feature_matrix, split_train_val_test

if not VALIDATED_CSV.exists():
    raw = download_benchmark_dataset()
    df = validate_and_normalize_dataset(raw)
else:
    df = pd.read_csv(VALIDATED_CSV)

print('Dataset shape:', df.shape)
print('Class distribution (0=benign, 1=phishing):')
print(df['label'].value_counts())
df.head()

### Dataset Quality Audit
After validating schemes (`http://` and `https://`) and removing 3 duplicate normalized URLs, the corpus contains **11,427** unique URLs (**5,715 benign**, **5,712 phishing**). Both classes include full paths and mixed HTTP/HTTPS schemes, avoiding artificial collection bias.

In [ ]:
X_df, y_ser, meta_df = build_feature_matrix(df)
print('Missing values per feature:', int(X_df.isna().sum().sum()))
X_df.describe().T[['mean', 'std', 'min', 'max']].head(15)

### Strict Leakage-Free Splitting & Model Comparison
We split the data into **70% train (7,996)**, **15% validation (1,714)**, and **15% test (1,714)** stratified by class **before** fitting any `SimpleImputer` or `StandardScaler` inside each Scikit-learn `Pipeline`.

In [ ]:
from train import run_training

metadata = run_training()
pd.DataFrame(metadata['validation_comparison']).T

### Held-Out Test Evaluation & Conclusion
On the validation split, **Random Forest** achieved the strongest F1-score (**0.8773**) and precision/recall balance over Logistic Regression (**0.7993**) and XGBoost (**0.8733**). Evaluated once on the held-out test set (1,714 URLs), Random Forest achieved:
- **Accuracy:** 86.58%
- **Precision:** 90.13%
- **Recall:** 82.13%
- **F1-Score:** 0.8594
- **ROC-AUC:** 0.9432

Because static URL lexical features cannot observe post-click webpage DOM content, the ML output is combined with multi-signal heuristics and external threat intelligence feeds in `backend/app/services/risk_aggregator.py`, while keeping `model_confidence` independent from the overall `0–100` risk score.